<a href="https://colab.research.google.com/github/LtFordo/SCY1101_Precios_Consumidor/blob/main/ProyectoProgramaci%C3%B3nCienciaDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/LtFordo/SCY1101_Precios_Consumidor

fatal: destination path 'SCY1101_Precios_Consumidor' already exists and is not an empty directory.


In [9]:
# Importación de las librerías a utilizar
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Se carga el dataset en su estado original para ejecutar posteriormente procesos de limpieza,
#validación y transformación de datos.
raw=pd.read_csv('https://raw.githubusercontent.com/LtFordo/SCY1101_Precios_Consumidor/main/data/precio_consumidor_arica_parinacota.csv')

raw['Region'].value_counts()

,count
Region,
Región de Arica y Parinacota,13753


In [10]:
# Respaldo del dataset original para trabajar sin reparos.
df = raw.copy()

In [11]:
# =================================================================
# FASE 1: AUDITORÍA DE INTEGRIDAD Y PROCEDENCIA
# =================================================================
# Justificación: La auditoría inicial permite caracterizar el dataset
# y detectar inconsistencias de tipos o esquemas antes de la manipulación.

try:
    print("--- Auditoría Inicial del Dataset ---")
    print(raw.info())
    # Verificación de valores imposibles y distribución inicial
    print("\n--- Estadísticas Descriptivas Crudas ---")
    print(raw.describe())
except Exception as e:
    print(f"Error en auditoría inicial: {e}")
df

--- Auditoría Inicial del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13753 entries, 0 to 13752
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Anio                     13053 non-null  float64
 1   Mes                      13050 non-null  float64
 2   Semana                   13046 non-null  float64
 3   Fecha inicio             12815 non-null  object 
 4   Fecha termino            12827 non-null  object 
 5   ID region                13061 non-null  float64
 6   Region                   13753 non-null  object 
 7   Sector                   13081 non-null  object 
 8   Tipo de punto monitoreo  13043 non-null  object 
 9   Grupo                    13047 non-null  object 
 10  Producto                 13076 non-null  object 
 11  Unidad                   13058 non-null  object 
 12  Precio minimo            12797 non-null  float64
 13  Precio maximo            12812 non-nul

,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Rango_precio
0,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Abastero,$/kilo,8499.0,8799.0,8599.000000,300.0
1,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Asado Carnicero,$/kilo,7999.0,7999.0,7999.000000,0.0
2,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Asado de tira,$/kilo,10299.0,10299.0,10299.000000,0.0
3,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Estomaguillo (Tapabarriga),$/kilo,7999.0,7999.0,7999.000000,0.0
4,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Ganso,$/kilo,8499.0,8799.0,8649.000000,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13748,2025.0,2.0,8.0,2025-02-17,2025-02-21,15.0,Región de Arica y Parinacota,Arica,Mercado Minorista,Hortalizas,Zanahoria|Sin especificar|Segunda,$/kilo,148990.0,NaN,700.000000,NaN
13749,2025.0,13.0,52.0,2025-12-22,2025-12-26,15.0,Región de Arica y Parinacota,Arica,Supermercado,Abarrotes y otros,Miel,$/envase 1 kilo,9590.0,10380.0,9985.000000,790.0
13750,2025.0,2.0,7.0,2025-02-10,2025-02-14,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne de Cerdo - Ave - Cordero,Pollo Pechuga Deshuesada,$/kilo,4899.0,6598.0,5439.142857,1699.0
13751,2025.0,4.0,18.0,2025-04-28,2025-05-02,15.0,Región de Arica y Parinacota,Arica,NaN,Hortalizas,Tomate|Larga vida|Segunda,$/kilo,1200.0,1300.0,1233.333333,100.0


In [12]:
# =================================================================
# FASE 2: LIMPIEZA PROFUNDA
# =================================================================
# Justificación: Se aplican múltiples técnicas de limpieza para eliminar
# el Bias (Sesgo) y asegurar que el modelo no procese datos inconsistentes.

try:
    # 1. Gestión de Duplicados: Eliminación de registros idénticos
    # que sesgan las métricas estadísticas.
    df.drop_duplicates(inplace=True)

    # 2. Corrección de Tipo de Dato: Conversión forzada a tipos numéricos y fechas.
    # Es mandatorio para permitir transformaciones avanzadas y cálculos matemáticos [7, 9].
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    # 3. Tratamiento de Outliers e Inconsistencias Lógicas:
    # Se eliminan valores imposibles (Mes > 12, Semana > 52) y precios negativos.
    df = df[(df['Mes'] <= 12) & (df['Semana'] <= 53)]
    df = df[df['Precio promedio'] >= 0]

    # Filtrado de outliers de error de digitación (Precios mínimos que exceden promedios)
    limite_max = df['Precio promedio'].max()
    df = df[df['Precio minimo'] <= limite_max]

    # 4. Normalización de Categóricos (Columna Region):
    # Eliminación de espacios y gestión de nulos representados como texto.
    df['Region'] = df['Region'].str.strip()
    valor_moda_region = df['Region'].mode()[0] # Extracción de valor escalar
    df['Region'] = df['Region'].replace('None', valor_moda_region)

    # Columna Sector
    df['Sector'] = (df['Sector'].replace('None', pd.NA))
    df['Sector'] = (df.groupby('Region')['Sector'].transform(lambda x: x.fillna(x.mode()[0])))


    # 5. Imputación Profesional de Nulos (Missing Values):
    # Se utiliza imputación lógica para no perder registros valiosos.
    # Precios faltantes se rellenan con el promedio del registro para mantener la integridad.
    df['Precio minimo'] = df['Precio minimo'].fillna(df['Precio promedio'])
    df['Precio maximo'] = df['Precio maximo'].fillna(df['Precio promedio'])

    # Categorías faltantes se etiquetan para auditoría posterior
    cols_cat = ['Sector', 'Tipo de punto monitoreo', 'Grupo', 'Unidad']
    df[cols_cat] = df[cols_cat].fillna('No especificado')

    # Barrido final para asegurar 0 nulos
    df.dropna(inplace=True)

    print("Fase de limpieza completada exitosamente.")

except Exception as e:
    print(f"Error crítico en el flujo de limpieza: {e}")
df

Fase de limpieza completada exitosamente.


,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Rango_precio
0,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Abastero,$/kilo,8499.0,8799.0,8599.000000,300.0
1,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Asado Carnicero,$/kilo,7999.0,7999.0,7999.000000,0.0
2,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Asado de tira,$/kilo,10299.0,10299.0,10299.000000,0.0
3,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Estomaguillo (Tapabarriga),$/kilo,7999.0,7999.0,7999.000000,0.0
4,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne bovina,Ganso,$/kilo,8499.0,8799.0,8649.000000,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13745,2025.0,12.0,50.0,2025-12-08,2025-12-12,15.0,Región de Arica y Parinacota,Arica,Feria libre,Frutas,Frutilla|Sin especificar|Primera,$/kilo,3000.0,3700.0,3350.000000,700.0
13746,2025.0,2.0,8.0,2025-02-17,2025-02-21,15.0,Región de Arica y Parinacota,Arica,Supermercado,Lácteos - Huevos - Margarinas,Leche Fluida Descremada,$/Caja de 1 Litro,960.0,1290.0,1105.250000,330.0
13750,2025.0,2.0,7.0,2025-02-10,2025-02-14,15.0,Región de Arica y Parinacota,Arica,Carnicería,Carne de Cerdo - Ave - Cordero,Pollo Pechuga Deshuesada,$/kilo,4899.0,6598.0,5439.142857,1699.0
13751,2025.0,4.0,18.0,2025-04-28,2025-05-02,15.0,Región de Arica y Parinacota,Arica,No especificado,Hortalizas,Tomate|Larga vida|Segunda,$/kilo,1200.0,1300.0,1233.333333,100.0


In [13]:
# =================================================================
# FASE 3: TRANSFORMACIÓN AVANZADA
# =================================================================
# Justificación: Se utiliza vectorización y escalamiento para mejorar
# el poder predictivo y normalizar la varianza de los datos.

try:
    # 1. Ingeniería de Características: Creación de columna con IVA mediante broadcasting.
    # La vectorización optimiza la memoria al evitar ciclos.
    df['Precio_IVA'] = df['Precio promedio'] * 1.19

    # 2. Codificación Categórica (One-Hot Encoding):
    # Transforma 'Region' en variables binarias procesables por modelos de IA.
    df_final = pd.get_dummies(df, columns=['Region'], prefix='Reg', dtype=int)

    # 3. Estandarización (Scaling):
    # Uso de StandardScaler para ajustar los precios a una media 0 y desviación 1.
    # Justificación: Esta técnica asume una distribución gaussiana y mitiga
    # el impacto de la magnitud de los precios en el análisis.
    scaler = StandardScaler()
    df_final['Precio_Promedio_Escalado'] = scaler.fit_transform(df_final[['Precio promedio']])

    print("Fase de transformación avanzada finalizada.")

except Exception as e:
    print(f"Error en transformaciones avanzadas: {e}")

df_final[['ID region', 'Sector']].value_counts()



Fase de transformación avanzada finalizada.


ID region  Sector  
15.0       Arica       7557
             Arica      156
             None        11
Name: count, dtype: int64

In [14]:
# =================================================================
# FASE 4: VALIDACIÓN Y EXPORTACIÓN
# =================================================================
# Justificación: La reproducibilidad exige que el archivo procesado se guarde
# en la estructura de carpetas definida para el proyecto (/data) [19-21].

try:
    print("--- Validación Final de Integridad ---")
    print(f"Registros totales: {len(df_final)}")
    print(f"Nulos remanentes: {df_final.isnull().sum().sum()}")

    # 1. Definición de la ruta de salida (Requisito de estructura de carpetas)
    # Se utiliza la ruta preexistente en la carpeta raíz para cumplir con
    # la organización jerárquica del repositorio (/data) [5, 6].
    ruta_salida = "SCY1101_Precios_Consumidor/data/precio_consumidor_2025_processed.csv"

    # 2. Exportación del Dataset Preparado
    # Justificación: El archivo se exporta en formato CSV sin índice para asegurar
    # la interoperabilidad y que el código corra "a la primera" en otros entornos [3, 7].
    df_final.to_csv(ruta_salida, index=False)

    print(f"Dataset final exportado exitosamente a: {ruta_salida}")

except Exception as e:
    # El manejo de excepciones es un requisito formal para garantizar la
    # robustez del flujo profesional de datos [1].
    print(f"Error en la fase final de validación o exportación: {e}")

try:
    print("--- Auditoría Inicial del Dataset ---")
    print(df_final.info())
    # Verificación de valores imposibles y distribución inicial
    print("\n--- Estadísticas Descriptivas Crudas ---")
    print(df_final.describe())
except Exception as e:
    print(f"Error en auditoría inicial: {e}")

df_final

--- Validación Final de Integridad ---
Registros totales: 7724
Nulos remanentes: 0
Dataset final exportado exitosamente a: SCY1101_Precios_Consumidor/data/precio_consumidor_2025_processed.csv
--- Auditoría Inicial del Dataset ---
<class 'pandas.core.frame.DataFrame'>
Index: 7724 entries, 0 to 13752
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Anio                              7724 non-null   float64       
 1   Mes                               7724 non-null   float64       
 2   Semana                            7724 non-null   float64       
 3   Fecha inicio                      7724 non-null   datetime64[ns]
 4   Fecha termino                     7724 non-null   datetime64[ns]
 5   ID region                         7724 non-null   float64       
 6   Sector                            7724 non-null   object        
 7   Tipo de punto monitoreo         

,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Rango_precio,Precio_IVA,Reg_Región de Arica y Parinacota,Precio_Promedio_Escalado
0,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Arica,Carnicería,Carne bovina,Abastero,$/kilo,8499.0,8799.0,8599.000000,300.0,10232.810000,1,1.902156
1,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Arica,Carnicería,Carne bovina,Asado Carnicero,$/kilo,7999.0,7999.0,7999.000000,0.0,9518.810000,1,1.693031
2,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Arica,Carnicería,Carne bovina,Asado de tira,$/kilo,10299.0,10299.0,10299.000000,0.0,12255.810000,1,2.494676
3,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Arica,Carnicería,Carne bovina,Estomaguillo (Tapabarriga),$/kilo,7999.0,7999.0,7999.000000,0.0,9518.810000,1,1.693031
4,2025.0,1.0,1.0,2024-12-30,2025-01-03,15.0,Arica,Carnicería,Carne bovina,Ganso,$/kilo,8499.0,8799.0,8649.000000,300.0,10292.310000,1,1.919583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13745,2025.0,12.0,50.0,2025-12-08,2025-12-12,15.0,Arica,Feria libre,Frutas,Frutilla|Sin especificar|Primera,$/kilo,3000.0,3700.0,3350.000000,700.0,3986.500000,1,0.072660
13746,2025.0,2.0,8.0,2025-02-17,2025-02-21,15.0,Arica,Supermercado,Lácteos - Huevos - Margarinas,Leche Fluida Descremada,$/Caja de 1 Litro,960.0,1290.0,1105.250000,330.0,1315.247500,1,-0.709729
13750,2025.0,2.0,7.0,2025-02-10,2025-02-14,15.0,Arica,Carnicería,Carne de Cerdo - Ave - Cordero,Pollo Pechuga Deshuesada,$/kilo,4899.0,6598.0,5439.142857,1699.0,6472.580000,1,0.800814
13751,2025.0,4.0,18.0,2025-04-28,2025-05-02,15.0,Arica,No especificado,Hortalizas,Tomate|Larga vida|Segunda,$/kilo,1200.0,1300.0,1233.333333,100.0,1467.666666,1,-0.665086


In [15]:
try:
    # 1. Eliminación de Duplicados: Evita sesgar las estadísticas del modelo [6, 9]
    df.drop_duplicates(inplace=True)

    # 2. Corrección de tipos: Conversión a numérico y datetime [10]
    # Usamos errors='coerce' para gestionar datos mal formateados como NaN
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    print("Corrección de formatos y eliminación de duplicados finalizada.")
except Exception as e:
    print(f"Error en corrección de tipos: {e}")

Corrección de formatos y eliminación de duplicados finalizada.
